## Prepping Data

In [1]:
import os

# Tell transformers not to use TensorFlow (we only need PyTorch here)
os.environ["TRANSFORMERS_NO_TF"] = "1"

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)

import transformers, sys, importlib

print("Python executable:", sys.executable)
print("Transformers version:", transformers.__version__)
print("TensorFlow present?", importlib.util.find_spec("tensorflow") is not None)
print("Keras present?", importlib.util.find_spec("keras") is not None)
print("tf_keras present?", importlib.util.find_spec("tf_keras") is not None)

/Users/chadadelman/anaconda3/envs/chad_env/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Python executable: /Users/chadadelman/anaconda3/envs/chad_env/bin/python
Transformers version: 4.57.3
TensorFlow present? True
Keras present? True
tf_keras present? True


In [2]:
#We will need the training data text summary pairs
%store -r train_paired_summaries


In [3]:
#create object to be passed in as batch
#Needs to be dictinionary of lists
#Will be only training data
text_list=[]
summary_list=[]

for v in train_paired_summaries.values():
    text_list.append(v[0])
    summary_list.append(v[1])

training_batch = dict()
training_batch["text"] = text_list
training_batch["summary"] = summary_list

training_batch_dataset = Dataset.from_dict(training_batch)

## Model Fine Tuning

In [7]:
import torch
from torch.utils.data import DataLoader

def run_model(
        test_text_list
        ,learning_rate = 5e-7
        ,batch_size = 2
        ,num_epochs = 2
        ,test_size = 0.4
        ,max_input_length = 64
        ,max_target_length = 32
        ,max_length = 32
        ,num_beams = 2
):
    # Simple train/validation split
    dataset = training_batch_dataset.train_test_split(test_size=test_size, seed=42)
    train_dataset = dataset["train"]
    eval_dataset = dataset["test"]

    model_name = "t5-small"

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

    print("Tokenizer type:", type(tokenizer))
    print("Model type:", type(model))

    def preprocess_function(batch):
        # T5 likes a task prefix, e.g. "summarize: "
        inputs = ["summarize: " + t for t in batch["text"]]
        model_inputs = tokenizer(
            inputs,
            max_length=max_input_length,
            truncation=True,
            padding="max_length",
        )

        labels = tokenizer(
            batch["summary"],
            max_length=max_target_length,
            truncation=True,
            padding="max_length",
        )

        model_inputs["labels"] = labels["input_ids"]
        return model_inputs

    tokenized_train = train_dataset.map(
        preprocess_function,
        batched=True,
        remove_columns=["text", "summary"],
    )

    tokenized_eval = eval_dataset.map(
        preprocess_function,
        batched=True,
        remove_columns=["text", "summary"],
    )

    data_collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        model=model,
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    # DataLoader for training
    train_dataloader = DataLoader(
        tokenized_train,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=data_collator,
    )

    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

    model.train()
    for epoch in range(num_epochs):
        total_loss = 0.0
        for step, batch in enumerate(train_dataloader):
            # Move batch to device
            batch = {k: v.to(device) for k, v in batch.items()}

            outputs = model(**batch)
            loss = outputs.loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

            if (step + 1) % 5 == 0:
                print(f"Epoch {epoch+1}, Step {step+1}, Loss: {loss.item():.4f}")

        avg_loss = total_loss / len(train_dataloader)
        print(f"Epoch {epoch+1} finished. Average loss: {avg_loss:.4f}")

    print("Training complete (manual loop).")

    model.eval()

    generated_text_list=[]

    for test_text in test_text_list:
        #summarize each item in test_text_list
        inputs = tokenizer(
            "summarize: " + test_text,
            return_tensors="pt",
            truncation=True,
            padding=True,
        ).to(device)

        with torch.no_grad():
            generated_ids = model.generate(
                **inputs,
                max_length=max_length,
                num_beams=num_beams,
            )

        generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
        generated_text_list.append(generated_text)
    
    return generated_text_list


In [ ]:
#Setting hyperparameters and testing
test_text_list = ["""FROM fairest creatures we desire increase,
That thereby beauty's rose might never die,
But as the riper should by time decease,
His tender heir might bear his memory:
But thou, contracted to thine own bright eyes,
Feed'st thy light'st flame with self-substantial fuel,
Making a famine where abundance lies,
Thyself thy foe, to thy sweet self too cruel.
Thou that art now the world's fresh ornament
And only herald to the gaudy spring,
Within thine own bud buriest thy content
And, tender churl, makest waste in niggarding.
Pity the world, or else this glutton be,
To eat the world's due, by the grave and thee.""",

"""Those hours, that with gentle work did frame
The lovely gaze where every eye doth dwell,
Will play the tyrants to the very same
And that unfair which fairly doth excel:
For never-resting time leads summer on
To hideous winter and confounds him there;
Sap cheque'd with frost and lusty leaves quite gone,
Beauty o'ersnow'd and bareness every where:
Then, were not summer's distillation left,
A liquid prisoner pent in walls of glass,
Beauty's effect with beauty were bereft,
Nor it nor no remembrance what it was:
But flowers distill'd though they with winter meet,
Leese but their show; their substance still lives sweet.
"""
]
             

learning_rate = 5e-7
batch_size = 2
num_epochs = 2
test_size = 0.4
max_input_length = 64
max_target_length = 32
max_length = 32
num_beams = 2

generated_text = run_model(
        test_text_list = test_text_list
        ,learning_rate = learning_rate
        ,batch_size = batch_size
        ,num_epochs = num_epochs
        ,test_size = test_size
        ,max_input_length = max_input_length
        ,max_target_length = max_target_length
        ,max_length = max_length
        ,num_beams = num_beams
)

print("SUMMARIES:", generated_text)

Tokenizer type: <class 'transformers.models.t5.tokenization_t5_fast.T5TokenizerFast'>
Model type: <class 'transformers.models.t5.modeling_t5.T5ForConditionalGeneration'>


Map:   0%|          | 0/454 [00:00<?, ? examples/s]

Map:   0%|          | 0/303 [00:00<?, ? examples/s]

Epoch 1, Step 5, Loss: 5.2807
Epoch 1, Step 10, Loss: 3.9836
Epoch 1, Step 15, Loss: 5.3135
Epoch 1, Step 20, Loss: 5.5714
Epoch 1, Step 25, Loss: 7.2354
Epoch 1, Step 30, Loss: 6.5876
Epoch 1, Step 35, Loss: 4.9973
Epoch 1, Step 40, Loss: 6.4324
Epoch 1, Step 45, Loss: 5.5237
Epoch 1, Step 50, Loss: 4.8672
Epoch 1, Step 55, Loss: 5.4524
Epoch 1, Step 60, Loss: 6.9757
Epoch 1, Step 65, Loss: 5.8101
Epoch 1, Step 70, Loss: 4.8864
Epoch 1, Step 75, Loss: 5.0115
Epoch 1, Step 80, Loss: 5.8473
Epoch 1, Step 85, Loss: 4.9274
Epoch 1, Step 90, Loss: 5.6866
Epoch 1, Step 95, Loss: 4.6347
Epoch 1, Step 100, Loss: 4.8961
Epoch 1, Step 105, Loss: 4.8386
Epoch 1, Step 110, Loss: 5.9138
Epoch 1, Step 115, Loss: 5.2040
Epoch 1, Step 120, Loss: 6.1151
Epoch 1, Step 125, Loss: 5.4301
Epoch 1, Step 130, Loss: 4.7251
Epoch 1, Step 135, Loss: 5.4025
Epoch 1, Step 140, Loss: 5.6818
Epoch 1, Step 145, Loss: 5.8212
Epoch 1, Step 150, Loss: 4.7710
Epoch 1, Step 155, Loss: 5.6972
Epoch 1, Step 160, Loss: 6.0